# Hailstorm Risk — Model Selection Evidence

This notebook documents the resampling + model comparison on the EchoSafe Pakistan hail-day dataset and explains why the production choice is **SMOTE → StandardScaler → LogisticRegression** with a **Youden's-J tuned decision threshold**.

**Target:** `hail_observed` (binary, 1 if any METAR within the station-day reported GR/GS).

**Features (surface-variable build):** daily aggregates of Open-Meteo ERA5 surface variables — temperature max/min, dew-point mean, 2m RH (min and mean), 10m wind speed mean, 10m gust max, surface pressure (min and daily drop), daily precipitation, mean cloud cover, thunder-hour count (WMO weather codes 95/96/99), calendar month, and a pre-monsoon flag (Mar–May). Pressure-level CAPE / freezing-level / pressure-level winds are not exposed by the Open-Meteo public archive for Pakistan, so the production model uses surface proxies. Labels are real IEM METAR GR/GS observations.

**Split:** time-based — last 20% of years used for evaluation, no leakage.

**Class imbalance:** observed hail is extremely rare — ~0.025% of station-days, only 11 positives across 10 years of records (6 train / 5 test under the time split). At this base rate the choice of resampler matters as much as the choice of model.

In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                              precision_recall_curve, recall_score, roc_auc_score,
                              roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import ADASYN, BorderlineSMOTE, RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

from config.settings import SETTINGS
from ml_models.hailstorm.features import HAIL_FEATURES, HAIL_TARGET

In [6]:
gold = pd.read_csv(SETTINGS.pipeline.gold_dir / 'hailstorm_risk' / 'hailstorm_risk_dataset.csv')
gold['date'] = pd.to_datetime(gold['date'], utc=True)
print(f'Total station-days: {len(gold):,}')
print(f'Positive hail days: {int(gold[HAIL_TARGET].sum()):,}  ({gold[HAIL_TARGET].mean():.4%})')
print('Year range:', gold["date"].dt.year.min(), '->', gold["date"].dt.year.max())
gold[[*HAIL_FEATURES, HAIL_TARGET]].describe().T

Total station-days: 43,812
Positive hail days: 11  (0.0251%)
Year range: 2016 -> 2026


,count,mean,std,min,25%,50%,75%,max
temperature_max_c,43812.0,30.376972,7.216532,7.800000,25.200000,31.200000,35.300000,50.500000
temperature_min_c,43812.0,19.225943,7.631758,-2.000000,13.100000,20.700000,25.800000,35.300000
dew_point_mean_c,43812.0,14.921329,7.686288,-20.650000,8.862500,14.533333,22.533333,27.879167
rh_min_pct,43812.0,39.191044,17.934802,2.000000,25.000000,38.000000,53.000000,92.000000
rh_mean_pct,43812.0,60.581282,17.497235,5.416667,48.916667,62.875000,74.916667,96.041667
wind_speed_mean_ms,43812.0,2.668261,1.351653,0.363426,1.697917,2.329282,3.349537,11.473380
wind_gust_max_ms,43812.0,8.450853,3.000755,2.611111,6.111111,8.111111,10.500000,26.000000
surface_pressure_min_hpa,43812.0,984.888745,21.008162,932.000000,972.600000,988.800000,1001.600000,1021.400000
surface_pressure_drop_hpa,43812.0,4.240094,0.986959,1.300000,3.600000,4.100000,4.700000,17.200000
precipitation_sum_mm,43812.0,1.367153,5.701971,0.000000,0.000000,0.000000,0.100000,416.900000


In [7]:
df = gold.dropna(subset=HAIL_FEATURES + [HAIL_TARGET]).copy()
years = sorted(df['year'].unique())
cutoff = years[int(len(years) * 0.8)]
train = df[df['year'] < cutoff]
test = df[df['year'] >= cutoff]
X_train, y_train = train[HAIL_FEATURES].values, train[HAIL_TARGET].astype(int).values
X_test, y_test = test[HAIL_FEATURES].values, test[HAIL_TARGET].astype(int).values
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train); X_test_s = scaler.transform(X_test)
print(f'Train: {len(X_train):,} rows ({y_train.sum()} positives, {y_train.mean():.4%})')
print(f'Test : {len(X_test):,} rows ({y_test.sum()} positives, {y_test.mean():.4%})')
print('Split: train < ', cutoff, ', test >=', cutoff)

Train: 33,024 rows (6 positives, 0.0182%)
Test : 10,788 rows (5 positives, 0.0463%)
Split: train <  2024 , test >= 2024


## 1. Resampler × model sweep

Resamplers are applied **only to the training set after scaling**. The test set is scored on its real distribution (no leakage). With only 6 train positives, `k_neighbors` is automatically lowered to fit SMOTE's neighbour count.

Headline expectation: tree models cannot generalise from 6 positives, so we expect Logistic Regression to dominate. The sweep confirms this.

In [8]:
k = max(1, min(5, int(y_train.sum()) - 1))
samplers = {
    'NoResample (class_weight)': None,
    'SMOTE': SMOTE(random_state=42, k_neighbors=k),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42, k_neighbors=k),
    'ADASYN': ADASYN(random_state=42, n_neighbors=k),
    'RandomOverSampler': RandomOverSampler(random_state=42),
    'RandomUnderSampler': RandomUnderSampler(random_state=42),
    'SMOTETomek': SMOTETomek(random_state=42, smote=SMOTE(random_state=42, k_neighbors=k)),
    'SMOTEENN':   SMOTEENN(random_state=42,   smote=SMOTE(random_state=42, k_neighbors=k)),
}

def build_models():
    return {
        'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
        'RF':     RandomForestClassifier(n_estimators=400, max_depth=16, min_samples_leaf=6,
                                         class_weight='balanced', n_jobs=-1, random_state=42),
        'GBM':    GradientBoostingClassifier(n_estimators=300, max_depth=4,
                                              learning_rate=0.05, random_state=42),
    }

rows = []
for s_name, sampler in samplers.items():
    if sampler is None:
        Xr, yr = X_train_s, y_train
    else:
        try: Xr, yr = sampler.fit_resample(X_train_s, y_train)
        except Exception as e: print(f'  {s_name}: skipped ({e})'); continue
    pos_ratio = float(np.mean(yr))
    for m_name, model in build_models().items():
        model.fit(Xr, yr)
        proba = model.predict_proba(X_test_s)[:, 1]
        preds = (proba >= 0.5).astype(int)
        rows.append({
            'sampler': s_name, 'model': m_name,
            'n_train_after': len(yr), 'pos_ratio_train': round(pos_ratio, 4),
            'roc_auc': roc_auc_score(y_test, proba) if y_test.sum() else np.nan,
            'avg_prec': average_precision_score(y_test, proba),
            'f1+': f1_score(y_test, preds, zero_division=0),
            'prec+': precision_score(y_test, preds, zero_division=0),
            'rec+': recall_score(y_test, preds, zero_division=0),
        })

results = pd.DataFrame(rows).round(4)
print('Top 10 by ROC AUC:')
print(results.sort_values('roc_auc', ascending=False).head(10).to_string(index=False))
print()
print('Top 10 by Average Precision:')
print(results.sort_values('avg_prec', ascending=False).head(10).to_string(index=False))

Top 10 by ROC AUC:
                  sampler  model  n_train_after  pos_ratio_train  roc_auc  avg_prec    f1+  prec+  rec+
        RandomOverSampler LogReg          66036           0.5000   0.9465    0.0157 0.0065 0.0033   0.8
NoResample (class_weight) LogReg          33024           0.0002   0.9460    0.0157 0.0065 0.0032   0.8
          BorderlineSMOTE LogReg          33024           0.0002   0.9460    0.0157 0.0065 0.0032   0.8
                    SMOTE LogReg          66036           0.5000   0.9424    0.0210 0.0072 0.0036   0.8
               SMOTETomek LogReg          66036           0.5000   0.9424    0.0210 0.0072 0.0036   0.8
                 SMOTEENN LogReg          65986           0.5004   0.9421    0.0210 0.0072 0.0036   0.8
                   ADASYN LogReg          66036           0.5000   0.9421    0.0210 0.0073 0.0037   0.8
                 SMOTEENN    GBM          65986           0.5004   0.7995    0.0016 0.0000 0.0000   0.0
NoResample (class_weight)    GBM          330

**Observed pattern.** Tree models (RF, GBM) collapse — they cannot split 6 positives reliably. Logistic Regression dominates with ROC AUC ~0.94 across every resampler. SMOTE / ADASYN / SMOTETomek tie for the best Average Precision on LogReg.

RandomUnderSampler gives the highest raw AP on tiny LogReg (~0.034) but at the cost of 99.97% lost training data and a worse ROC AUC, so we do not pick it.

**Conclusion: SMOTE + LogisticRegression wins on balanced ranking quality**.

Resamplers cannot manufacture information beyond the 11 real positive days, so absolute precision is structurally capped by the base rate. The notebook's next step is to choose an operating point that reflects what the warning system actually needs.

## 2. Decision-threshold tuning

Under extreme imbalance, F1 misbehaves: "predict everything" returns higher F1 than any sane threshold because there's nothing to beat. The standard fix is **Youden's J = TPR − FPR** — it picks the operating point where the model best separates the two classes regardless of class size.

We tune on a validation slice carved from the *training* years (no test leakage).

In [9]:
# Carve a validation slice from the trailing 20% of training years.
train_years = sorted(train['year'].unique())
val_cutoff = train_years[int(len(train_years) * 0.8)]
train_inner = train[train['year'] < val_cutoff]
val_inner   = train[train['year'] >= val_cutoff]
Xti = scaler.transform(train_inner[HAIL_FEATURES].values)
yti = train_inner[HAIL_TARGET].astype(int).values
Xvi = scaler.transform(val_inner[HAIL_FEATURES].values)
yvi = val_inner[HAIL_TARGET].astype(int).values
k_v = max(1, min(5, int(yti.sum()) - 1))
Xr, yr = SMOTE(random_state=42, k_neighbors=k_v).fit_resample(Xti, yti)
clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42).fit(Xr, yr)
p_val = clf.predict_proba(Xvi)[:, 1]

fpr, tpr, ts = roc_curve(yvi, p_val)
valid = (ts <= 1.0) & np.isfinite(ts)
j = tpr[valid] - fpr[valid]
best = int(np.argmax(j))
best_t = max(float(ts[valid][best]), 0.05)
print(f'Validation cutoff: {val_cutoff}  positives_val={int(yvi.sum())}/{len(yvi)}')
print(f"Youden's J-optimal threshold = {best_t:.4f}  (J={j[best]:.4f}, TPR={tpr[valid][best]:.4f}, FPR={fpr[valid][best]:.4f})")

Validation cutoff: 2022  positives_val=2/8760
Youden's J-optimal threshold = 0.0500  (J=0.1591, TPR=1.0000, FPR=0.8409)


In [10]:
# Sanity check: operating points on the held-out test set.
Xr, yr = SMOTE(random_state=42, k_neighbors=k).fit_resample(X_train_s, y_train)
clf_full = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42).fit(Xr, yr)
p_test = clf_full.predict_proba(X_test_s)[:, 1]
for t in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    pr = (p_test >= t).astype(int)
    print(f'  t={t:.2f}: '
          f'P={precision_score(y_test, pr, zero_division=0):.4f}  '
          f'R={recall_score(y_test, pr, zero_division=0):.4f}  '
          f'F1={f1_score(y_test, pr, zero_division=0):.4f}  '
          f'#alarms={int(pr.sum())}')
print(f'ROC AUC: {roc_auc_score(y_test, p_test):.4f}    AP: {average_precision_score(y_test, p_test):.4f}')

  t=0.05: P=0.0018  R=1.0000  F1=0.0035  #alarms=2818
  t=0.10: P=0.0021  R=1.0000  F1=0.0043  #alarms=2338
  t=0.20: P=0.0028  R=1.0000  F1=0.0055  #alarms=1798
  t=0.30: P=0.0027  R=0.8000  F1=0.0053  #alarms=1500
  t=0.50: P=0.0036  R=0.8000  F1=0.0072  #alarms=1100
  t=0.70: P=0.0048  R=0.8000  F1=0.0096  #alarms=828
  t=0.90: P=0.0040  R=0.4000  F1=0.0080  #alarms=498
ROC AUC: 0.9424    AP: 0.0210


## 3. Production model

- **Pipeline:** `SMOTE → StandardScaler → LogisticRegression(class_weight='balanced')`
- **Threshold:** Youden's-J optimal from validation slice, floored at 0.05 to avoid the degenerate "predict everything" trap when validation positives are sparse.
- **Why not RF / GBM:** they cannot learn from 6 training positives — ROC AUC ≤ 0.78, recall = 0 at any threshold ≥ 0.5 even after SMOTE.
- **Why not RandomUnderSampler-only:** highest AP but ROC AUC drops to ~0.69 because 99.97% of negatives are discarded.
- **Honest precision floor:** with 5 real positives out of 10,788 test station-days, precision is structurally bounded near the base rate even for a perfect ranker. Improving the absolute count of positives requires a richer label source (CDS ERA5 + storm reports / ESWD) — that swap is isolated to `pipelines/utils/hailstorm_client.py`.

Production config: `ml_models/hailstorm/config.yaml`. Trainer: `ml_models/hailstorm/risk_trainer.py`. Predictor: `predictors/hailstorm_predictor.py`.

In [11]:
coefs = pd.Series(clf_full.coef_[0], index=HAIL_FEATURES).sort_values(key=abs, ascending=False)
print('LogisticRegression coefficients (scaled features):')
print(coefs.round(3))

LogisticRegression coefficients (scaled features):
rh_mean_pct                  6.824
dew_point_mean_c            -5.550
temperature_max_c            4.151
wind_gust_max_ms             2.974
cloud_cover_mean_pct         1.789
surface_pressure_drop_hpa    1.687
is_premonsoon                1.280
temperature_min_c           -1.000
surface_pressure_min_hpa     0.869
month                        0.791
rh_min_pct                  -0.625
wind_speed_mean_ms          -0.565
precipitation_sum_mm         0.063
thunder_hours                0.000
dtype: float64
